# 04 - Hot-Spot Stress Method Benchmark

Validate the structural hot-spot stress method per IIW recommendations.

**Type a (linear extrapolation):**
$$\sigma_{hs} = 1.67 \cdot \sigma(0.4t) - 0.67 \cdot \sigma(1.0t)$$

**Type b (3-point extrapolation):**
$$\sigma_{hs} = 3 \cdot \sigma(5\text{mm}) - 3 \cdot \sigma(15\text{mm}) + \sigma(25\text{mm})$$

In [ ]:
from weldfatigue.fatigue.hotspot_stress import HotSpotStressAssessment
from weldfatigue.fatigue.sn_curve import SNCurve
import numpy as np

## Type a Extrapolation Validation

In [ ]:
assessor = HotSpotStressAssessment(fat_class=100, material="steel")

# Verify extrapolation formula
s_04t = 150.0  # stress at 0.4t from weld toe
s_10t = 120.0  # stress at 1.0t from weld toe

hs_type_a = assessor.extrapolate_type_a(s_04t, s_10t)
expected = 1.67 * s_04t - 0.67 * s_10t

print(f"σ(0.4t) = {s_04t:.1f} MPa")
print(f"σ(1.0t) = {s_10t:.1f} MPa")
print(f"σ_hs (Type a) = {hs_type_a:.2f} MPa")
print(f"Expected      = {expected:.2f} MPa")
print(f"Match: {abs(hs_type_a - expected) < 0.01}")

## Type b Extrapolation Validation

In [ ]:
s_5 = 180.0   # stress at 5mm
s_15 = 140.0  # stress at 15mm
s_25 = 110.0  # stress at 25mm

hs_type_b = assessor.extrapolate_type_b(s_5, s_15, s_25)
expected_b = 3 * s_5 - 3 * s_15 + s_25

print(f"σ(5mm)  = {s_5:.1f} MPa")
print(f"σ(15mm) = {s_15:.1f} MPa")
print(f"σ(25mm) = {s_25:.1f} MPa")
print(f"σ_hs (Type b) = {hs_type_b:.2f} MPa")
print(f"Expected      = {expected_b:.2f} MPa")

## Hot-Spot Fatigue Assessment

In [ ]:
result = assessor.evaluate(stress_range=90.0, num_cycles=2_000_000)

print(f"Method: {result.method}")
print(f"FAT class: {result.fat_class}")
print(f"Allowable cycles: {result.allowable_cycles:.2e}")
print(f"Safety factor: {result.safety_factor:.3f}")
print(f"Status: {result.status}")

## Visualize: Effect of Hot-Spot FAT Class

In [ ]:
import matplotlib.pyplot as plt

fig, ax = plt.subplots(figsize=(10, 6))

for fc in [90, 100, 112]:
    sn = SNCurve(fat_class=fc, material_type="steel")
    n_v, s_v = sn.get_curve_points()
    ax.loglog(n_v, s_v, linewidth=2, label=f"FAT {fc}")

ax.axhline(y=90, color='red', linestyle=':', alpha=0.7, label='Operating σ_hs=90 MPa')
ax.set_xlabel("N cycles", fontsize=12)
ax.set_ylabel("Δσ [MPa]", fontsize=12)
ax.set_title("Hot-Spot Method - S-N Curves for Typical FAT Classes", fontsize=14)
ax.legend()
ax.grid(True, which='both', alpha=0.3)
plt.tight_layout()
plt.show()